# 16.3 Generics — `TypeVar`, PEP 695 and Variance

**Prerequisites:** 16.1, 16.2, 4.4 Decorators, 5.1 OOPs  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- The problem generics solve — and why `Any` is not the answer
- 🔴 **PEP 695** (3.12): `def first[T](...)` and `class Stack[T]` — the modern syntax
- Bounds (`T: Job`) vs constraints (`T: (int, float)`)
- The `type` statement for aliases
- 🔴 **Variance** — why `list[BuildJob]` is *not* a `list[Job]`, demonstrated
- How PEP 695 **infers** variance from how you use the parameter
- The practical rule: accept `Sequence`, return `list`
- `ParamSpec` — typing a decorator without destroying the signature (**4.4**)
- Legacy `TypeVar` syntax, for reading code written before 3.12

---

## The problem

Write a function that returns the first item of a sequence. What is its return type?

```python
def first(items):          # no types: caller learns nothing
def first(items: list) -> Any:      # Any: caller learns nothing, and 16.1 showed why that spreads
def first(items: list[int]) -> int | None:    # honest, but only works for int
```

You would need one copy per element type. **Generics** let you say *"whatever type went in,
that is the type that comes out"* — a **relationship** between the argument and the return, not
a fixed type.

```
   first( list[int]  )  ─────>  int | None
   first( list[str]  )  ─────>  str | None
   first( list[Job]  )  ─────>  Job | None
          ▲                     ▲
          └── one definition ───┘
              the type flows through
```

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py163_"))


def write(name, source):
    """Write a script into the scratch directory and return its name."""
    (WORK / name).write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return name


def mypy(name, source=None, *flags):
    """Type-check a file with mypy and return its report."""
    if source is not None:
        write(name, source)
    done = subprocess.run(
        [sys.executable, "-m", "mypy", name,
         "--cache-dir", str(WORK / ".mypy_cache"),
         "--no-color-output", "--no-error-summary", *flags],
        cwd=WORK, capture_output=True, text=True,
        encoding="utf-8", errors="replace", timeout=300)
    report = (done.stdout + done.stderr).strip() or "(mypy found nothing to report)"
    return (f"$ mypy {name} {' '.join(flags)}".rstrip() + "\n" + "-" * 68 + "\n"
            + report + "\n" + "-" * 68 + f"\nexit code: {done.returncode}")


def python(name):
    done = subprocess.run([sys.executable, name], cwd=WORK, capture_output=True,
                          text=True, encoding="utf-8", errors="replace", timeout=60)
    return (f"$ python {name}\n" + "-" * 68 + "\n"
            + (done.stdout + done.stderr).strip() + "\n" + "-" * 68
            + f"\nexit code: {done.returncode}")


print("scratch:", WORK)

## 🔴 PEP 695 — the modern syntax (3.12+)

Python 3.12 introduced a syntax that needs **no import and no separate declaration**: square
brackets after the name.

```
def first[T](items: Sequence[T]) -> T | None:
        ─┬─               ─┬─       ─┬─
         │                 │         └─ ...comes back out here
         │                 └─ used here...
         └─ declare the type parameter
```

Everything before 3.12 needed `T = TypeVar("T")` at module level first. You will still meet
that constantly — it is at the end of this notebook — but new code should use the bracket form.

In [ ]:
print(mypy("generic_fn.py", r"""
    from collections.abc import Sequence


    def first[T](items: Sequence[T]) -> T | None:
        return items[0] if items else None


    def pairs[K, V](mapping: dict[K, V]) -> list[tuple[K, V]]:
        return list(mapping.items())


    reveal_type(first([1, 2, 3]))
    reveal_type(first(["eu", "us"]))
    reveal_type(first([1.5]))

    reveal_type(pairs({"queued": 3, "running": 1}))

    # The relationship is enforced in both directions:
    region: str = first(["eu"])            # 🔴 str | None is not str
"""))

One definition, three different revealed types — `int | None`, `str | None`,
`float | None`. And the last line shows the checker still holds you to the `| None`.

This is exactly the exercise **4.5** left you with: *"Write `def first(items: Sequence[T]) -> T
| None` for an empty-safe first element."*

## Generic classes

The same brackets work on a class. `Stack[T]` means "a stack of some particular type", and the
checker tracks which one.

In [ ]:
print(mypy("generic_cls.py", r"""
    class Stack[T]:
        def __init__(self) -> None:
            self._items: list[T] = []

        def push(self, item: T) -> None:
            self._items.append(item)

        def pop(self) -> T:
            return self._items.pop()

        def peek(self) -> T | None:
            return self._items[-1] if self._items else None


    jobs: Stack[str] = Stack()
    jobs.push("build-1")
    reveal_type(jobs.pop())
    reveal_type(jobs.peek())

    jobs.push(42)                       # 🔴 wrong element type

    counts = Stack[int]()               # explicit parameterisation also works
    counts.push(1)
    reveal_type(counts.pop())
"""))

## Bounds and constraints

Sometimes `T` cannot be *anything* — you need to call a method on it, or restrict it to a
handful of types. There are two mechanisms, and they behave differently.

| Form | Means | `T` becomes |
|---|---|---|
| `def f[T: Job](...)` | **bound** — any subtype of `Job` | the *actual* subtype passed |
| `def f[T: (int, float)](...)` | **constrained** — exactly one of these | `int` or `float`, never a subclass |

🔴 A **bound** preserves the subclass; a **constraint** collapses to the listed type. Reach for
a bound almost always — constraints exist mainly for things like `AnyStr`, and `AnyStr`
itself is **deprecated since 3.13**: new code writes a PEP 695 constraint
(`[S: (str, bytes)]`) or plain `str` overloads instead.

In [ ]:
print(mypy("bounds.py", r"""
    class Job:
        priority: int = 0

    class BuildJob(Job):
        def build(self) -> str:
            return "built"


    def highest[J: Job](jobs: list[J]) -> J:               # BOUND by Job
        return max(jobs, key=lambda job: job.priority)     # .priority is available


    def largest[N: (int, float)](values: list[N]) -> N:    # CONSTRAINED
        return max(values)


    builds = [BuildJob(), BuildJob()]
    reveal_type(highest(builds))          # keeps BuildJob - the bound preserves it
    highest(builds).build()               # so this is allowed

    reveal_type(largest([1, 2]))
    reveal_type(largest([1.5, 2.5]))

    highest([object()])                   # 🔴 not a Job
    largest(["a", "b"])                   # 🔴 not int or float
"""))

`highest(builds)` was revealed as **`BuildJob`**, not `Job` — so `.build()` is
available. That is the whole value of a bound: the constraint is enforced *and* the precise
type survives.

## The `type` statement

PEP 695 also gave aliases a real syntax. Before, an alias was just an assignment and the
checker had to guess your intent.

In [ ]:
print(mypy("aliases.py", r"""
    from collections.abc import Callable

    type JobId = str
    type Seconds = float
    type Registry[T] = dict[JobId, list[T]]           # a GENERIC alias
    type RetryPolicy = Callable[[int], Seconds]


    def schedule(registry: Registry[str], policy: RetryPolicy) -> Seconds:
        return policy(len(registry))


    jobs: Registry[str] = {"build-1": ["queued", "running"]}
    reveal_type(jobs)

    schedule(jobs, lambda attempt: 2.0 ** attempt)
    schedule({"build-1": [1, 2]}, lambda attempt: 1.0)     # 🔴 list[int], not list[str]
"""))
print()
print(python(write("alias_rt.py", r"""
    type JobId = str
    print("a type alias is a real object:", type(JobId).__name__)
    print("  value            :", JobId)
    print("  its target       :", JobId.__value__)
""")))

🔴 `type X = ...` creates a **`TypeAliasType` object**, and it is **lazy** — the
right-hand side is not evaluated until `__value__` is touched. That is why a `type` alias can
refer to something defined later in the file, where a plain assignment could not.

## 🔴 Variance — the hard part

Here is the question that trips up everyone:

> `BuildJob` is a subclass of `Job`. So is `list[BuildJob]` a `list[Job]`?

The intuitive answer is yes. **The correct answer is no**, and the reason is worth seeing
rather than being told.

In [ ]:
print(mypy("whyinvariant.py", r"""
    class Job: pass


    class BuildJob(Job):
        def build(self) -> str:
            return "built"


    class DeployJob(Job):
        pass


    def sabotage(jobs: list[Job]) -> None:
        jobs.append(DeployJob())          # perfectly legal: a DeployJob IS a Job


    builds: list[BuildJob] = [BuildJob()]
    sabotage(builds)                      # 🔴 if this were allowed...
    builds[-1].build()                    # ...this would explode at runtime
"""))

Read the demonstration, not just the error. If `list[BuildJob]` were
accepted as `list[Job]`, then `sabotage` — which is **correctly typed and obviously fine** —
could append a `DeployJob` into a list the caller believes holds only `BuildJob`s. The next
`.build()` call would fail at runtime.

So `list` must be **invariant**: `list[BuildJob]` and `list[Job]` are unrelated types. Note that
mypy tells you the fix in its own note: *"Consider using `Sequence` instead, which is
covariant."*

### The three kinds

| Variance | If `BuildJob` is a `Job`, then… | Because the parameter appears… | Examples |
|---|---|---|---|
| **Covariant** | `Box[BuildJob]` **is** a `Box[Job]` | only as **output** (you read) | `Sequence`, `Iterable`, `tuple`, `frozenset` |
| **Contravariant** | `Box[Job]` **is** a `Box[BuildJob]` | only as **input** (you write) | `Callable` argument types |
| **Invariant** | neither | as **both** | `list`, `dict`, `set` |

The mnemonic that actually sticks: **read-only is covariant, write-only is contravariant,
read-write is invariant.**

### PEP 695 works this out for you

Before 3.12 you declared variance by hand — `TypeVar("T_co", covariant=True)` — and getting it
wrong was easy. With the bracket syntax the checker **infers** it from how the parameter is
used.

In [ ]:
print(mypy("variance.py", r"""
    class Job: pass
    class BuildJob(Job): pass


    class ReadOnlyBox[T]:                 # T appears only as OUTPUT
        def __init__(self, item: T) -> None:
            self._item = item

        def get(self) -> T:
            return self._item


    class MutableBox[T]:                  # T appears as INPUT and output
        def __init__(self, item: T) -> None:
            self._item = item

        def get(self) -> T:
            return self._item

        def put(self, item: T) -> None:
            self._item = item


    build_ro: ReadOnlyBox[BuildJob] = ReadOnlyBox(BuildJob())
    build_mut: MutableBox[BuildJob] = MutableBox(BuildJob())

    as_job_ro: ReadOnlyBox[Job] = build_ro       # inferred COVARIANT - allowed
    as_job_mut: MutableBox[Job] = build_mut      # inferred INVARIANT - error
"""))

Two classes, one difference — `MutableBox` has a `put` method — and the
checker worked out the variance of each without being told. **Adding a setter silently changes
what your generic class is compatible with**, which is a genuinely useful thing to know.

## 🔴 The practical rule

You will use variance far more often than you think about it, in one specific decision:

> **Accept the most general type you can. Return the most specific type you can.**
>
> In practice: take `Sequence[T]` / `Iterable[T]` / `Mapping[K, V]` as **parameters**,
> return `list[T]` / `dict[K, V]`.

Taking `list` when you only iterate makes your function needlessly picky — and the caller gets
the confusing invariance error rather than anything about their actual mistake.

In [ ]:
print(mypy("params.py", r"""
    from collections.abc import Iterable, Sequence


    class Job: pass
    class BuildJob(Job): pass


    def count_picky(jobs: list[Job]) -> int:        # 🔴 needlessly restrictive
        return len(jobs)


    def count_general(jobs: Sequence[Job]) -> int:  # covariant - accepts subtypes
        return len(jobs)


    def count_widest(jobs: Iterable[Job]) -> int:   # accepts generators, sets, anything
        return sum(1 for _ in jobs)


    builds: list[BuildJob] = [BuildJob()]

    count_general(builds)                    # fine
    count_widest(builds)                     # fine
    count_widest(job for job in builds)      # a generator works too
    count_picky(builds)                      # 🔴 the invariance error
"""))

Three functions that do the same thing; only the picky one rejects a
perfectly reasonable argument. `Iterable` is the widest — it even accepts a generator (**4.3**).

## `ParamSpec` — typing a decorator

**4.4** built decorators and noted that `functools.wraps` preserves the *name* and *docstring*.
It does not preserve the **signature** as far as a type checker is concerned — the naive
annotation `Callable[..., Any]` throws away every argument type.

`ParamSpec` captures "whatever parameters the wrapped function had". In PEP 695 syntax it is
`**P`:

```
def retry[**P, R](func: Callable[P, R]) -> Callable[P, R]:
          ─┬─ ─┬─
           │   └─ the return type
           └─ the whole parameter list, as one unit
```

In [ ]:
print(mypy("paramspec.py", r"""
    import functools
    from collections.abc import Callable


    def retry[**P, R](func: Callable[P, R]) -> Callable[P, R]:
        @functools.wraps(func)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
            return func(*args, **kwargs)
        return wrapper


    def logged(func: Callable[..., object]) -> Callable[..., object]:
        # 🔴 the naive version: every argument type is thrown away
        return func


    @retry
    def fetch(url: str, timeout: float = 1.0) -> bytes:
        return b""


    @logged
    def fetch_untyped(url: str, timeout: float = 1.0) -> bytes:
        return b""


    reveal_type(fetch)
    reveal_type(fetch_untyped)

    fetch("https://example.com", timeout=2.0)
    fetch(123)                            # 🔴 caught through the decorator
    fetch_untyped(123)                    # not caught - the signature is gone
"""))

`fetch` kept its full signature — `def (url: str, timeout: float =) -> bytes`
— **through the decorator**, so passing `123` was caught. `fetch_untyped` degraded to
`Callable[..., object]` and the same mistake sailed through.

> `Concatenate` is the companion for decorators that *add* a parameter — for example an
> `@inject_connection` that supplies the first argument. `Callable[Concatenate[Conn, P], R]`
> means "the wrapped function takes a `Conn` plus whatever else".

## Legacy `TypeVar`, for reading older code

Everything above is 3.12+. Most code you read today still uses the pre-695 form, and the two
interoperate freely.

| Modern (3.12+) | Legacy |
|---|---|
| `def first[T](...)` | `T = TypeVar("T")` then `def first(...)` |
| `class Stack[T]:` | `class Stack(Generic[T]):` |
| `def f[T: Job](...)` | `TypeVar("T", bound=Job)` |
| `def f[T: (int, float)](...)` | `TypeVar("T", int, float)` |
| `def d[**P, R](...)` | `P = ParamSpec("P")`, `R = TypeVar("R")` |
| `type Alias = X` | `Alias: TypeAlias = X` or plain `Alias = X` |
| variance **inferred** | `TypeVar("T_co", covariant=True)` by hand |

In [ ]:
print(mypy("legacy.py", r"""
    from typing import Generic, TypeVar

    T = TypeVar("T")
    J = TypeVar("J", bound="Job")


    class Job:
        priority: int = 0


    class OldStack(Generic[T]):
        def __init__(self) -> None:
            self._items: list[T] = []

        def push(self, item: T) -> None:
            self._items.append(item)

        def pop(self) -> T:
            return self._items.pop()


    def old_first(items: list[T]) -> T | None:
        return items[0] if items else None


    reveal_type(old_first([1, 2]))

    stack: OldStack[str] = OldStack()
    stack.push("build-1")
    reveal_type(stack.pop())
    stack.push(42)                         # 🔴 same checking, older spelling
"""))

Identical behaviour, more ceremony. 🔴 The one real difference is
**variance**: with `Generic[T]` you must declare `covariant=True` yourself and it is easy to get
wrong, whereas PEP 695 infers it. That alone is a good reason to prefer the new syntax in code
you own.

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **Expecting `list[BuildJob]` to be a `list[Job]`.** It is not, and the demonstration in this notebook shows why allowing it would be unsound.
2. 🔴 **Taking `list[X]` as a parameter when you only read it.** You get the invariance error and your caller has no idea why. Take `Sequence[X]` or `Iterable[X]`.
3. **Using a constraint where you wanted a bound.** `T: (int, float)` collapses subclasses; `T: Job` preserves them.
4. **Reaching for `Any` because generics look hard.** `Any` disables checking everywhere the value travels (**16.1**).
5. **Declaring `TypeVar(..., covariant=True)` on a class with a setter.** It is unsound, and it is exactly the mistake PEP 695's inference removes.
6. **Typing a decorator as `Callable[..., Any]`.** Every argument type is discarded — use `ParamSpec` (**4.4**).
7. **Adding a setter to a generic class** without realising it changes the class from covariant to invariant, breaking callers.
8. **Using one `TypeVar` for two unrelated things.** Two appearances of `T` in a signature mean *the same type*; if they need not match, use `T` and `U`.
9. **Parameterising with a value rather than a type** — `Stack[3]` is not a thing.

## Best Practices

- Use the PEP 695 bracket syntax in new code; it needs no import and infers variance.
- Accept the widest type you can (`Iterable`, `Sequence`, `Mapping`) and return the most specific (`list`, `dict`).
- Prefer a bound (`T: Job`) to a constraint — it preserves the caller's exact type.
- Use `type X = ...` for aliases; it is lazy and unambiguous.
- Use `ParamSpec` for every decorator that forwards its arguments.
- Let variance be inferred rather than declared — and notice when adding a setter changes it.
- Name type parameters for what they are: `T` for a general item, but `K`/`V`, `R`, `JobT` when it aids reading.
- `reveal_type` at each call site is the fastest way to check a generic does what you meant.

## Practice Exercises

Try these before moving on.

1. Write `def last[T](items: Sequence[T]) -> T | None` and confirm with `reveal_type` that it specialises for `int`, `str` and a class of your own.
2. Build `class Cache[K, V]` with `get`, `set` and `keys`. Is it covariant or invariant in `V`? Prove your answer with an assignment.
3. 🔴 Reproduce the `sabotage` demonstration, then change `list[Job]` to `Sequence[Job]` and explain in one sentence why the error disappears *and* the code is still safe.
4. Write `def widest[T: Job](jobs: Sequence[T]) -> T` and call it with a `list` of a subclass. Confirm the subclass survives the call.
5. Take a decorator from **4.4** and type it with `ParamSpec`. Then retype it as `Callable[..., Any]` and find what stops being checked.
6. Convert a legacy `Generic[T]` class into PEP 695 syntax. Did the declared variance match what the checker now infers?
7. 🔴 Write a generic `Result[T, E]` type with `ok` and `error` variants, and a function that narrows it (**16.2**). What does `reveal_type` show in each branch?
8. **Interview question:** why is `list` invariant when `Sequence` is covariant, and what goes wrong if you get it backwards?

---

## Version notes

| Version | Change |
|---|---|
| **3.13** | `TypeVar` gained defaults (PEP 696): `class Box[T = str]` |
| **3.12** | 🔴 **PEP 695** — `def f[T]()`, `class C[T]`, `type X = ...`, and **inferred variance**. The whole of this notebook's modern half |
| **3.11** | `Self` (**16.4**); variadic generics (`TypeVarTuple`, `*Ts`) |
| **3.10** | `ParamSpec` and `Concatenate` added to `typing` |
| **3.9** | `list[int]` / `dict[str, int]` builtins usable directly in annotations |
| **3.7** | `from __future__ import annotations` — makes all annotations strings, which is how pre-3.10 code writes modern syntax |

## Where next

| Notebook | Covers |
|---|---|
| **16.4** | protocols, structural typing, `Self`, `@overload` |
| **16.5** | typing real code: classes, decorators, async, third-party stubs |
| **16.6** | adopting types in an existing codebase |

## Related

- **16.1** — why `Any` is not an acceptable substitute for a generic
- **16.2** — narrowing, which is how you use a generic union safely
- **4.3 Generators** — `Iterable`, the widest parameter type
- **4.4 Decorators** — what `ParamSpec` exists to preserve
- **5.1 OOPs** — subclassing, which is what variance is about
- **14 Data Structure and Algorithm** — `Stack[T]`, `Queue[T]`, `Tree[T]`: the natural home for generics